# 02 - Enriquecimiento, Unificación y Limpieza
(solo hice un poco de limpieza)

In [1]:
# 1. Inicialización de PySpark y credenciales Snowflake
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month

load_dotenv('../.env')

spark = (
    SparkSession.builder
    .appName("Enriquecimiento_Unificacion")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.4,net.snowflake:snowflake-jdbc:3.14.4"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

sfOptions = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW", "RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}
print("PySpark y credenciales listos.")

PySpark y credenciales listos.


In [5]:
# Ver el número de núcleos (slots) disponibles
print(f"Número de núcleos: {spark.sparkContext.defaultParallelism}")

Número de núcleos: 24


### Mostrar Columnas de ambas Tablas


In [2]:
# Cargar tablas usando Snowflake Connector for Spark
df_yellow = spark.read.format("snowflake") \
  .options(**sfOptions) \
  .option("dbtable", "YELLOW_TRIPS_RAW") \
  .load()

df_green = spark.read.format("snowflake") \
  .options(**sfOptions) \
  .option("dbtable", "GREEN_TRIPS_RAW") \
  .load()

print("1. Columnas de YELLOW_TRIPS_RAW:")
print(df_yellow.columns)
print("\n2. Columnas de GREEN_TRIPS_RAW:")
print(df_green.columns)


1. Columnas de YELLOW_TRIPS_RAW:
['VENDOR_ID', 'PICKUP_DATETIME', 'DROPOFF_DATETIME', 'PASSENGER_COUNT', 'TRIP_DISTANCE', 'RATE_CODE_ID', 'STORE_AND_FWD_FLAG', 'PU_LOCATION_ID', 'DO_LOCATION_ID', 'PAYMENT_TYPE', 'FARE_AMOUNT', 'EXTRA', 'MTA_TAX', 'TIP_AMOUNT', 'TOLLS_AMOUNT', 'IMPROVEMENT_SURCHARGE', 'TOTAL_AMOUNT', 'CONGESTION_SURCHARGE', 'AIRPORT_FEE', 'RUN_ID', 'SOURCE_YEAR', 'SOURCE_MONTH', 'SERVICE_TYPE', 'INGESTED_AT_UTC']

2. Columnas de GREEN_TRIPS_RAW:
['VENDOR_ID', 'PICKUP_DATETIME', 'DROPOFF_DATETIME', 'STORE_AND_FWD_FLAG', 'RATE_CODE_ID', 'PU_LOCATION_ID', 'DO_LOCATION_ID', 'PASSENGER_COUNT', 'TRIP_DISTANCE', 'FARE_AMOUNT', 'EXTRA', 'MTA_TAX', 'TIP_AMOUNT', 'TOLLS_AMOUNT', 'EHAIL_FEE', 'IMPROVEMENT_SURCHARGE', 'TOTAL_AMOUNT', 'PAYMENT_TYPE', 'TRIP_TYPE', 'CONGESTION_SURCHARGE', 'RUN_ID', 'SOURCE_YEAR', 'SOURCE_MONTH', 'SERVICE_TYPE', 'INGESTED_AT_UTC']


### Unificación y Limpieza de Calidad de Datos


In [3]:
# Unificación de tablas permitiendo columnas faltantes (Ej. GREEN tiene ehail_fee que YELLOW no)
df_unificado = df_yellow.unionByName(df_green, allowMissingColumns=True)

# Aplicamos las reglas de limpieza lógicamente
df_limpio = df_unificado.filter(
    # Regla 1: El mes y año de pickup_datetime debe coincidir con el registro fuente original
    (year(col("pickup_datetime")) == col("source_year")) &
    (month(col("pickup_datetime")) == col("source_month")) &
    # Regla 2: El dropoff no puede ser anterior al pickup (viajes en el tiempo inválidos)
    (col("dropoff_datetime") >= col("pickup_datetime"))
)

print("Filtros de limpieza aplicados correctamente al DataFrame. Listo para explorar o materializar.")
# Puedes correr df_limpio.show(5) para echarle un vistazo a los datos limpios.
df_limpio.show(5)

Filtros de limpieza aplicados correctamente al DataFrame. Listo para explorar o materializar.
+---------+--------------------+--------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+-----------+------------+------------+--------------------+---------+---------+
|VENDOR_ID|     PICKUP_DATETIME|    DROPOFF_DATETIME|PASSENGER_COUNT|TRIP_DISTANCE|RATE_CODE_ID|STORE_AND_FWD_FLAG|PU_LOCATION_ID|DO_LOCATION_ID|PAYMENT_TYPE|FARE_AMOUNT|EXTRA|MTA_TAX|TIP_AMOUNT|TOLLS_AMOUNT|IMPROVEMENT_SURCHARGE|TOTAL_AMOUNT|CONGESTION_SURCHARGE|AIRPORT_FEE|       RUN_ID|SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|     INGESTED_AT_UTC|EHAIL_FEE|TRIP_TYPE|
+---------+--------------------+--------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+--------